In [ ]:
import argparse
import json
import logging
import contextlib
from copy import deepcopy
from pathlib import Path

from tqdm import tqdm
import time

import numpy as np
import torch

# scdp imports (same as your test script)
from torch.utils.data import Subset
from scdp.common.pyg import DataLoader
from scdp.data.dataset import LmdbDataset
from scdp.data.datamodule import worker_init_fn
from scdp.model.module import ChgLightningModule

# PySCF for AO evaluation
from pyscf import gto


In [ ]:
def get_probe_chunks(n_probes, max_n_probe_per_pass):
    """
    Copied/simplified from your test module. 
    n_probes: torch tensor of length batch_size
    returns: n_pass (int), n_per_pass (list of tensors), probes_to_process (list of index tensors)
    """
    batch_size = len(n_probes)
    n_per_pass = []
    probes_to_process = []
    n_probes = torch.clone(n_probes)
    probe_indices = torch.arange(n_probes.sum(), device=n_probes.device)
    
    n_pass = int(((n_probes).sum() / max_n_probe_per_pass).ceil().long().item())
    pass_start_index = 0
    pass_end_index = 0
    for _ in range(n_pass):
        current_pass = torch.zeros(batch_size, dtype=torch.long, device=n_probes.device)
        current_load = 0

        for i in range(batch_size):
            if n_probes[i] == 0:
                continue
            max_points_for_job = (max_n_probe_per_pass - current_load)
            points_to_process = torch.min(torch.tensor([n_probes[i], max_points_for_job], device=n_probes.device))
            current_load += points_to_process
            current_pass[i] = points_to_process
            pass_end_index += int(points_to_process)
            n_probes[i] -= points_to_process
            if current_load >= max_n_probe_per_pass:
                break
        n_per_pass.append(current_pass)
        probes_to_process.append(probe_indices[pass_start_index:pass_end_index])
        pass_start_index = pass_end_index
        pass_end_index = pass_start_index

    return n_pass, n_per_pass, probes_to_process


def _try_get_attr(data_obj, candidates):
    """Try several attribute names and return the first one that exists (value)."""
    for name in candidates:
        if hasattr(data_obj, name):
            return getattr(data_obj, name)
    raise AttributeError(f"None of {candidates} found on data object. Inspect dataset class.")


def build_pyscf_mol_from_sample(sample, basis="def2-svp", unit="Angstrom"):
    """
    Build a PySCF Mol from a sample (Data object) — tries several common attribute names.
    Returns a built pyscf.gto.M object.
    """
    coords = _try_get_attr(sample, ["atom_coords", "coords", "pos", "atom_pos"])
    if isinstance(coords, torch.Tensor):
        coords = coords.cpu().numpy()
    atom_types = None
    for cand in ["atom_types", "z", "atomic_numbers", "atom_numbers", "nums", "atomic_num"]:
        if hasattr(sample, cand):
            atom_types = getattr(sample, cand)
            break
    if atom_types is None and hasattr(sample, "atom_symbols"):
        atom_types = sample.atom_symbols
    if atom_types is None:
        raise AttributeError("Could not infer atom types from sample. Please adapt the script to your dataset fields.")

    if isinstance(atom_types, torch.Tensor):
        atom_types = atom_types.cpu().numpy()

    # Convert numeric atomic numbers to element symbols if needed
    if np.issubdtype(np.array(atom_types).dtype, np.integer):
        from pyscf.data import elements
        syms = [elements.symbol(int(z)) for z in atom_types]
        atom_types = syms
    else:
        atom_types = [str(s) for s in atom_types]

    atom_str = "\n".join(f"{sym} {x:.9f} {y:.9f} {z:.9f}"
                         for sym, (x, y, z) in zip(atom_types, coords))
    mol = gto.M(atom=atom_str, basis=basis, unit=unit, verbose=0)
    mol.build(0, 0)
    return mol


def eval_ao_on_coords(mol, coords_np):
    """
    Evaluate contracted AO values on coords_np (N,3) using PySCF.
    Returns (N, n_ao) array.
    """
    return mol.eval_gto("GTOval", coords_np)


def project_from_grid(Phi, ao_coeffs):
    """
    Given Phi (N_grid x n_ao) and ao_coeffs (n_ao) return rho_recon (N_grid).
    If ao_coeffs has batch or other dims, we try to handle common formats.
    """
    coeffs = np.asarray(ao_coeffs)
    if coeffs.ndim > 1:
        if coeffs.shape[0] == 1:
            coeffs = coeffs.reshape(-1)
        else:
            # guess: collapse leading batch/orbital dims by summing last axis
            coeffs = np.sum(coeffs, axis=tuple(range(coeffs.ndim - 1))).reshape(-1)
    rho = Phi @ coeffs
    return rho


In [1]:
import numpy as np
from pyscf import gto

# -----------------------
# 1. Define H2O molecule and basis
# -----------------------
mol = gto.Mole()
mol.atom = """
O  0.000000  0.000000  0.000000
H  0.000000  0.757160  0.586260
H  0.000000 -0.757160  0.586260
"""
# Small basis to keep the test simple
mol.basis = 'sto-3g'
mol.build()

# -----------------------
# 2. Compute overlap matrix S
# -----------------------
S = mol.intor('int1e_ovlp')
print("Overlap matrix S shape:", S.shape)

# -----------------------
# 3. Create a synthetic density
# -----------------------
# Number of basis functions
nbasis = S.shape[0]

# True coefficients for our synthetic rho
c_true = np.random.rand(nbasis)

# We'll define rho(r) = sum c_true_nu * omega_nu(r)
# Now compute b_mu = sum_nu c_true_nu * S_{mu,nu}
b = S @ c_true

# -----------------------
# 4. Recover coefficients from overlaps
# -----------------------
c_recovered = np.linalg.solve(S, b)

# -----------------------
# 5. Compare results
# -----------------------
print("True coefficients:")
print(c_true)
print("Recovered coefficients:")
print(c_recovered)
print("Max abs diff:", np.max(np.abs(c_true - c_recovered)))

# -----------------------
# 6. Optional: Orthonormalization test
# -----------------------
# Löwdin orthonormalization
eigvals, eigvecs = np.linalg.eigh(S)
S_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
omega_ortho_coeffs = S_inv_sqrt @ c_true
print("Orthonormalized coeffs:", omega_ortho_coeffs)


Overlap matrix S shape: (7, 7)
True coefficients:
[0.33528073 0.8752242  0.36329458 0.33528846 0.62235445 0.66643474
 0.12497501]
Recovered coefficients:
[0.33528073 0.8752242  0.36329458 0.33528846 0.62235445 0.66643474
 0.12497501]
Max abs diff: 5.551115123125783e-16
Orthonormalized coeffs: [ 0.22891078  0.87903912  0.36329458  0.2478946   0.6347864   0.37376676
 -0.1754439 ]


In [ ]:
import numpy as np
from pyscf import gto, scf, dft

# -----------------------
# 1. Build molecule
# -----------------------
mol = gto.Mole()
mol.atom = """
O  0.000000  0.000000  0.000000
H  0.000000  0.757160  0.586260
H  0.000000 -0.757160  0.586260
"""
mol.basis = 'sto-3g'
mol.build()

# -----------------------
# 2. Run SCF to get density
# -----------------------
mf = dft.RKS(mol)
mf.xc = 'pbe'   # Use PBE for fun; can be 'lda', 'hf', etc.
mf.kernel()

# -----------------------
# 3. Get AO values on a numerical grid
# -----------------------
grids = dft.gen_grid.Grids(mol)
grids.level = 10  # grid density; increase for accuracy
grids.build()

# coords: (ngrids, 3), weights: (ngrids,)
coords = grids.coords
weights = grids.weights

# Evaluate all AOs on the grid
ao_values = dft.numint.eval_ao(mol, coords)  # shape (ngrids, nbasis)

# -----------------------
# 4. Get the *actual* density on the grid
# -----------------------
dm = mf.make_rdm1()
rho = dft.numint.eval_rho(mol, ao_values, dm)  # (ngrids,)

# -----------------------
# 5. Compute b_mu = ∫ rho(r) * ω_mu(r) dr
# -----------------------
b = np.einsum('g,gp,g->p', rho, ao_values, weights)

# -----------------------
# 6. Compute S analytically
# -----------------------
S = mol.intor('int1e_ovlp')

# -----------------------
# 7. Solve for coefficients: S c = b
# -----------------------
c_recovered = np.linalg.solve(S, b)

# -----------------------
# 8. Reconstruct density from recovered c
# -----------------------
rho_rec = np.dot(ao_values, c_recovered)  # (ngrids,)

# -----------------------
# 9. Compare densities
# -----------------------
diff = rho_rec - rho
int_abs_error = np.sum(np.abs(diff) * weights)
int_rho = np.sum(rho * weights)
int_rho_rec = np.sum(rho_rec * weights)

print("Electron count from SCF density:", int_rho)
print("Electron count from reconstructed density:", int_rho_rec)
print("Integrated absolute error:", int_abs_error)
print("Max pointwise error:", np.max(np.abs(diff)))


converged SCF energy = -75.2255044094244
Electron count from SCF density: 10.00000000012004
Electron count from reconstructed density: 12.911277463312954
Integrated absolute error: 9.329644597618113
Max pointwise error: 120.39601991314348


In [6]:
import numpy as np
from pyscf import gto, scf, dft

# -----------------------
# Define helper: primitive Gaussian
# -----------------------
def gaussian(r, alpha):
    """Normalized 3D s-type Gaussian: exp(-alpha * r^2)"""
    norm = (2 * alpha / np.pi)**(3/4)  # normalization for s-type
    return norm * np.exp(-alpha * r**2)

# -----------------------
# 1. Define molecule for DFT (H2O)
# -----------------------
mol = gto.Mole()
mol.atom = """
O  0.000000  0.000000  0.000000
H  0.000000  0.757160  0.586260
H  0.000000 -0.757160  0.586260
"""
mol.basis = 'sto-3g'
mol.build()

# -----------------------
# 2. Run SCF to get density
# -----------------------
mf = dft.RKS(mol)
mf.xc = 'pbe'
mf.kernel()

# -----------------------
# 3. Build integration grid
# -----------------------
grids = dft.gen_grid.Grids(mol)
grids.level = 5
grids.build()
coords = grids.coords   # (ngrids, 3)
weights = grids.weights # (ngrids,)

# -----------------------
# 4. Evaluate DFT density on the grid
# -----------------------
ao_values = dft.numint.eval_ao(mol, coords)
dm = mf.make_rdm1()
rho = dft.numint.eval_rho(mol, ao_values, dm)  # (ngrids,)

# -----------------------
# 5. Define custom Gaussian basis functions (Fu-style)
# -----------------------
atom_coords = mol.atom_coords()
n_atoms = atom_coords.shape[0]

# Let's place *one* s-type Gaussian on each atom (alpha = 0.5)
alphas = [0.5] * n_atoms
basis_functions = []

for R, alpha in zip(atom_coords, alphas):
    rvec = coords - R
    rnorm = np.linalg.norm(rvec, axis=1)
    basis_functions.append(gaussian(rnorm, alpha))

basis_functions = np.array(basis_functions).T  # shape (ngrids, nbasis)
nbasis = basis_functions.shape[1]

# -----------------------
# 6. Compute overlap matrix S numerically
# -----------------------
S = np.einsum("gi,gj,g->ij", basis_functions, basis_functions, weights)

# -----------------------
# 7. Compute b vector
# -----------------------
b = np.einsum("g,gi,g->i", rho, basis_functions, weights)

# -----------------------
# 8. Solve for coefficients
# -----------------------
c_recovered = np.linalg.solve(S, b)

# -----------------------
# 9. Reconstruct density from coefficients
# -----------------------
rho_rec = basis_functions @ c_recovered

# -----------------------
# 10. Compare results
# -----------------------
electron_count_true = np.sum(rho * weights)
electron_count_rec = np.sum(rho_rec * weights)
int_abs_error = np.sum(np.abs(rho - rho_rec) * weights)

print("True electron count:", electron_count_true)
print("Reconstructed electron count:", electron_count_rec)
print("Integrated absolute error:", int_abs_error)
print("Coefficients:", c_recovered)


converged SCF energy = -75.2255044094243
True electron count: 9.999999989467627
Reconstructed electron count: 13.790820380555008
Integrated absolute error: 11.320374425653059
Coefficients: [ 2.68948166 -0.31161621 -0.31161621]


In [11]:
import numpy as np
import torch

from pyscf import gto, dft
from scdp.model.basis_set import get_basis_set, transform_basis_set, aug_etb_for_basis
from scdp.model.gtos import GTOs

# -----------------------
# 1) Build molecule & run DFT
# -----------------------
mol = gto.Mole()
mol.atom = """
O  0.000000  0.000000  0.000000
H  0.000000  0.757160  0.586260
H  0.000000 -0.757160  0.586260
"""
mol.basis = 'sto-3g'
mol.build()

mf = dft.RKS(mol)
mf.xc = 'pbe'
mf.kernel()

# -----------------------
# 2) Build integration grid & DFT density
# -----------------------
grids = dft.gen_grid.Grids(mol)
grids.level = 7
grids.build()
coords = grids.coords
weights = grids.weights

ao_vals = dft.numint.eval_ao(mol, coords)
dm = mf.make_rdm1()
rho = dft.numint.eval_rho(mol, ao_vals, dm)

# -----------------------
# 3) Construct SCDP GTO basis
# -----------------------
# Define basis parameters similar to SCDP hyperparams
dft_basis_set = 'def2-svp'   # you can choose another
unique_atom_types = [mol.atom_symbol(i) for i in range(mol.natm)]
unique_atom_numbers = [mol.atom_charge(i) for i in range(mol.natm)]

basis_set = transform_basis_set(get_basis_set(dft_basis_set))

# Example: no augmentation, no uncontracting
gto_dict = {}
for Z in unique_atom_numbers:
    gto_dict[str(Z)] = GTOs(**basis_set[Z], cutoff=10.0)  # cutoff in bohr

# -----------------------
# 4) Evaluate SCDP GTO basis functions on the grid
# -----------------------
# GTOs are PyTorch modules, so convert coords to torch
coords_t = torch.tensor(coords, dtype=torch.float32)  # (G, 3)
basis_list = []

for atom_idx, Z in enumerate(unique_atom_numbers):
    center = mol.atom_coords()[atom_idx]
    center_t = torch.tensor(center, dtype=torch.float32).unsqueeze(0)
    # Evaluate this atom's GTOs at all grid points
    vals = gto_dict[str(Z)](
        probe_coords=coords_t,
        atom_coords=center_t,
        n_probes=torch.tensor([coords_t.shape[0]]),
        n_atoms=torch.tensor([1]),
        coeffs=None,           # we just want pure basis functions
        expo_scaling=None,
        pbc=False,
        cell=None
    )  # shape (G, nbasis_atom)
    basis_list.append(vals.detach().numpy())

B = np.hstack(basis_list)  # (G, nbasis_total)

# -----------------------
# 5) Fit coefficients by weighted least squares
# -----------------------
sqrtW = np.sqrt(weights)[:, None]
A_w = sqrtW * B
y_w = sqrtW[:, 0] * rho
c_fit, *_ = np.linalg.lstsq(A_w, y_w, rcond=None)

# -----------------------
# 6) Reconstruct density & compare
# -----------------------
rho_rec = B @ c_fit
int_abs_error = np.sum(np.abs(rho - rho_rec) * weights)

print("Integrated abs error:", int_abs_error)
print("Electron count true:", np.sum(rho * weights))
print("Electron count rec :", np.sum(rho_rec * weights))
#print("Fitted coefficients:", c_fit)


converged SCF energy = -75.2255044094243
Integrated abs error: 4.121401029998077
Electron count true: 10.000000002675366
Electron count rec : 10.601778549335574
